# **CHAINS In LangChain**

A Chain is a pipeline: each step's output feeds directly into the next step's input.
LangChain uses the `|` operator (LCEL — LangChain Expression Language) to wire steps together.

In [1]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser  # converts AIMessage → plain string

load_dotenv()

if os.environ.get("GROQ_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("GROQ_API_KEY not found")

llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


Bro API KEY Variable exists


# **First Chain**

Three independent pieces that each do one job.
Manually, you'd call them in sequence. With a chain, `|` does it for you.

In [2]:
# TASK 1 — Prompt template
# Defines the conversation shape. {input} is the user's question placeholder.
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("human",  "{input}")
])


In [3]:
# TASK 2 — The LLM itself
# Receives a filled ChatPromptValue, calls the Groq API, returns an AIMessage
llm_groq = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


In [4]:
# TASK 3 — Output parser
# StrOutputParser pulls .content out of the AIMessage → gives you a plain Python string
# Without this, the chain would return an AIMessage object (with metadata)
str_parser = StrOutputParser()


### **Manual Invocation** (for understanding what the chain does internally)

In [5]:
# Step-by-step without the | operator, so you can see each transformation:

# Step 1: fill the template — produces a ChatPromptValue (list of messages)
template = prompt_template.invoke({"input": "What is the capital of France?"})

# Step 2: send to LLM — produces an AIMessage
res = llm_groq.invoke(template)

# Step 3: extract the text — produces a plain string
final_result = str_parser.invoke(res)
final_result


'The capital of France is Paris.'

### **Chain Invocation** — the clean LCEL way

In [8]:
# The | operator creates a RunnableSequence under the hood
# Data flows left-to-right:  prompt → llm → parser
# .invoke() kicks off the whole pipeline with a single dict
chain = prompt_template | llm_groq | str_parser

chain.invoke({"input": "What is the capital of india?"})


'The capital of India is New Delhi.'

In [7]:
from langchain_core.runnables import RunnableSequence

# RunnableSequence is the explicit version of the | syntax — same result
# Useful if you're building chains programmatically (e.g. from a list of steps)
chain_1 = RunnableSequence(prompt_template, llm_groq, str_parser)

# Note: RunnableSequence.invoke() accepts a raw string when the first step
# has a single-variable template (it fills that variable automatically)
chain_1.invoke("What is the capital of France?")


'The capital of France is Paris.'